# Fine-tuning YOLO11s-cls no domínio REAL (fotos do celular) — v1 (500 fotos)

**Notebook self-contained** — rode **FT-0 (bootstrap)** primeiro: carrega o modelo base
do Roboflow do Drive (`soja_yolo11s_best.pt`) e define `crop_single_grain`, `CLASS_NAMES`.

### Fonte das fotos: pasta do Drive
As fotos reais de fine-tuning vêm de uma pasta no Drive (`REAL_SRC`, na FT-0), com **uma
subpasta por classe** (nome completo `Broken soybeans/` ou curto `broken/`). Meta: **~100
fotos por classe = 500**. Captura **manual**, no **mesmo setup do uso real** (fundo, luz e
celular que serão usados na demo) — é isso que mata o domain shift.

### Ordem
FT-0 (bootstrap) → FT-1 (monta train/val/test) → FT-2 (baseline) → FT-3 (fine-tune) →
FT-4 (antes×depois val+test + matriz) → FT-5 (salva no Drive se o **val** melhorar).

### Honestidade
Com ~500 fotos e split 70/15/15, o **val** (~15/classe) seleciona o modelo e o **test**
(~15/classe, nunca visto no treino) é o número honesto de relatório. A fronteira mais
ambígua continua sendo **skin-damaged × broken** (ambos têm dano de superfície).

In [ ]:
# FT-0 — BOOTSTRAP (rode SEMPRE que abrir num runtime novo / depois de restart)
#
# Self-contained: carrega o modelo base do Roboflow salvo no Drive
# (soja_yolo11s_best.pt, gerado pela Célula 8 do train_yolo.ipynb) e redefine
# CLASS_NAMES / crop. As fotos REAIS de fine-tuning vêm de uma PASTA DO DRIVE
# (ver REAL_SRC) — captura manual, ~100 fotos por classe (meta: 500).
!pip -q install ultralytics

import os, shutil, random, glob, tempfile, pathlib, json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from PIL import Image
from ultralytics import YOLO

from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive'

# >>> CONFIG — pasta no Drive com as fotos reais, UMA subpasta por classe. <<<
# As subpastas podem usar o nome completo ('Broken soybeans') OU o curto ('broken').
#   soja_correcoes/
#     Broken soybeans/   (ou broken/)
#     Immature soybeans/ (ou immature/)
#     Intact soybeans/   (ou intact/)
#     Skin-damaged soybeans/ (ou skin-damaged/)
#     Spotted soybeans/  (ou spotted/)
REAL_SRC = '/content/drive/MyDrive/soja_correcoes'

CLASS_NAMES = [
    'Broken soybeans', 'Immature soybeans', 'Intact soybeans',
    'Skin-damaged soybeans', 'Spotted soybeans',
]
SHORT = {
    'Broken soybeans':       'broken',
    'Immature soybeans':     'immature',
    'Intact soybeans':       'intact',
    'Skin-damaged soybeans': 'skin-damaged',
    'Spotted soybeans':      'spotted',
}
short_names = [SHORT[c] for c in CLASS_NAMES]
idx_of = {name: i for i, name in enumerate(CLASS_NAMES)}

def crop_single_grain(arr):
    """Mesmo recorte do app.py / Célula 7b: maior contorno via Otsu -> bounding box."""
    h, w = arr.shape[:2]
    gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
    blurred = cv2.GaussianBlur(gray, (7, 7), 0)
    _, thresh = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel, iterations=2)
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN,  kernel, iterations=1)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return arr
    largest = max(contours, key=cv2.contourArea)
    ratio = cv2.contourArea(largest) / (h * w)
    if ratio <= 0.02 or ratio >= 0.95:
        return arr
    x, y, bw, bh = cv2.boundingRect(largest)
    pad = max(15, int(min(bw, bh) * 0.12))
    return arr[max(0, y-pad):min(h, y+bh+pad), max(0, x-pad):min(w, x+bw+pad)]

# --- Carrega o modelo base do Roboflow salvo no Drive ---
BASE_PT = f'{SAVE_DIR}/soja_yolo11s_best.pt'
assert os.path.exists(BASE_PT), (
    f'\n❌ NÃO achei {BASE_PT}.\n'
    '   É o modelo base do Roboflow, salvo pela CÉLULA 8 do train_yolo.ipynb.\n'
    '   Rode o train_yolo.ipynb (células 1, 2, 3, 4 e 8) pra regenerá-lo no Drive.'
)
model = YOLO(BASE_PT)                       # modelo Roboflow = ponto de partida do FT
assert [model.names[i] for i in range(len(CLASS_NAMES))] == CLASS_NAMES, \
    'Ordem das classes do modelo carregado diverge de CLASS_NAMES!'
print('✅ Modelo Roboflow carregado de', BASE_PT)

# --- Confere a pasta de fotos reais no Drive ---
assert os.path.isdir(REAL_SRC), (
    f'\n❌ NÃO achei a pasta {REAL_SRC}.\n'
    '   Crie-a no Drive com uma subpasta por classe e suba as fotos reais.'
)
print('✅ Bootstrap pronto — pasta real:', REAL_SRC, '— pode rodar FT-1.')

In [ ]:
# FT-1 — Montar dataset de fine-tuning a partir das fotos REAIS do Drive
#         Recorta cada foto (1 grão) e faz split estratificado train/val/test.
REAL_DST = '/content/soja_real'
if os.path.exists(REAL_DST):
    shutil.rmtree(REAL_DST)
rng = random.Random(42)
IMG_EXTS_SET = {'.jpg', '.jpeg', '.png', '.webp'}
VAL_FRAC, TEST_FRAC = 0.15, 0.15   # 70/15/15 com ~100 fotos/classe -> ~15 val + 15 test

def _resolve_dir(cls):
    """Acha a subpasta da classe (nome completo OU curto)."""
    for cand in (os.path.join(REAL_SRC, cls), os.path.join(REAL_SRC, SHORT[cls])):
        if os.path.isdir(cand):
            return cand
    return None

summary = {}
for cls in CLASS_NAMES:
    srcdir = _resolve_dir(cls)
    assert srcdir, (f'❌ pasta não encontrada p/ "{cls}". Esperado '
                    f'{REAL_SRC}/{cls}/  ou  {REAL_SRC}/{SHORT[cls]}/')
    fs = sorted(p for p in glob.glob(os.path.join(srcdir, '*'))
                if pathlib.Path(p).suffix.lower() in IMG_EXTS_SET)
    rng.shuffle(fs)
    n = len(fs)
    n_val  = max(1, round(n * VAL_FRAC))  if n >= 3 else 0
    n_test = max(1, round(n * TEST_FRAC)) if n >= 4 else 0
    val_set  = set(fs[:n_val])
    test_set = set(fs[n_val:n_val + n_test])
    for p in fs:
        split = 'val' if p in val_set else ('test' if p in test_set else 'train')
        arr  = np.array(Image.open(p).convert('RGB'))
        crop = crop_single_grain(arr)
        outdir = os.path.join(REAL_DST, split, cls)
        os.makedirs(outdir, exist_ok=True)
        Image.fromarray(crop).save(os.path.join(outdir, pathlib.Path(p).stem + '.jpg'), quality=95)
    summary[SHORT[cls]] = (n - n_val - n_test, n_val, n_test)

print('classe        -> (train, val, test)')
total = [0, 0, 0]
for k, v in summary.items():
    print(f'  {k:>13}: {v}')
    total = [a + b for a, b in zip(total, v)]
print(f'  {"TOTAL":>13}: {tuple(total)}  (={sum(total)} fotos)')

# --- alerta de desbalanceamento (classe minoritária << majoritária no train) ---
trains = [v[0] for v in summary.values()]
if min(trains) and max(trains) > 2 * min(trains):
    print('\n⚠️  classes desbalanceadas no train (maior > 2x menor). Tente nivelar a '
          'captura (~100/classe) — augmentation forte ajuda, mas não resolve sozinho.')
if sum(total) < 250:
    print('\nℹ️  menos de 250 fotos no total — dá pra rodar, mas a meta é ~500 pra um val/test sólidos.')

In [ ]:
# FT-2 — Baseline: o modelo do Roboflow (ANTES do fine-tuning) no domínio real
import glob

def eval_on_split(yolo_model, split):
    paths, true = [], []
    for cls in CLASS_NAMES:
        for p in sorted(glob.glob(os.path.join(REAL_DST, split, cls, '*.jpg'))):
            paths.append(p); true.append(idx_of[cls])
    if not paths:
        return None, None, None
    imgs = [Image.open(p).convert('RGB') for p in paths]
    pred = [idx_of[yolo_model.names[int(r.probs.top1)]]
            for r in yolo_model.predict(imgs, imgsz=224, verbose=False)]
    true, pred = np.array(true), np.array(pred)
    return (true == pred).mean(), true, pred

base_tr_acc,   _, _ = eval_on_split(model, 'train')
base_val_acc,  _, _ = eval_on_split(model, 'val')
base_test_acc, _, _ = eval_on_split(model, 'test')
print('BASELINE (Roboflow, sem fine-tuning) no domínio real:')
print(f'  train: {base_tr_acc:.1%}')
print(f'  val:   {base_val_acc:.1%}')
print(f'  test:  {base_test_acc:.1%}' if base_test_acc is not None else '  test:  (sem split de test)')

In [ ]:
# FT-3 — Fine-tuning: parte do modelo Roboflow, congela o backbone, augmentation forte
#
# FREEZE controla quanto descongela:
#   freeze=10 -> treina SÓ a cabeça (Classify). Mais seguro, mexe pouco no domain shift.
#   freeze=9  -> treina o último bloco + cabeça. Adapta mais features (bom p/ fundo/luz).
#   freeze=8  -> mais capacidade; com ~500 fotos passa a ser viável sem decorar.
#
# Com ~500 fotos (vs 57 antes): batch maior (gradiente menos ruidoso), menos épocas
# e patience menor (converge mais rápido com mais dados). Augmentation forte continua
# sendo a alavanca principal contra o domain shift (fundo cinza / iluminação variada).
FREEZE = 9   # com dataset bem nivelado e >=80/classe, pode testar freeze=8

START_PT = BASE_PT   # ponto de partida = pesos do treino Roboflow (do Drive)

ft = YOLO(START_PT)
ft_results = ft.train(
    data=REAL_DST,
    epochs=60,
    imgsz=224,
    batch=32,           # ~500 fotos: batch maior estabiliza o treino
    freeze=FREEZE,
    lr0=0.0008,         # baixo p/ não destruir o aprendido, um tico acima do probe de 57
    patience=15,
    seed=42,
    # --- augmentation forte ---
    hsv_h=0.02, hsv_s=0.7, hsv_v=0.6,   # cor/brilho: robustez a iluminacao
    degrees=15, flipud=0.5, fliplr=0.5, # geometria: grao nao tem orientacao fixa
    translate=0.1, scale=0.5, erasing=0.4,
    project='/content/runs_soja',
    name='yolo11s_finetune',
    exist_ok=True,
    verbose=True,
)
print('\nFine-tuning concluido. save_dir:', ft.trainer.save_dir)

In [ ]:
# FT-4 — Antes x Depois no domínio real (val + test) + matriz de confusão
from sklearn.metrics import classification_report, confusion_matrix

new_tr_acc,   _, _                = eval_on_split(ft, 'train')
new_val_acc,  val_true, val_pred  = eval_on_split(ft, 'val')
new_test_acc, test_true, test_pred = eval_on_split(ft, 'test')

print('=== Domínio real: ANTES (Roboflow) x DEPOIS (fine-tuned) ===')
print(f'  train: {base_tr_acc:.1%}  ->  {new_tr_acc:.1%}')
print(f'  val:   {base_val_acc:.1%}  ->  {new_val_acc:.1%}   (usado p/ selecionar/gate)')
if new_test_acc is not None:
    print(f'  test:  {base_test_acc:.1%}  ->  {new_test_acc:.1%}   (headline honesto, não usado no treino)')

gap = new_tr_acc - new_val_acc
print(f'\n  gap train-val depois: {gap:.1%}  ', end='')
print('(gap grande = overfitting)' if gap > 0.15 else '(gap saudável)')

# relatório + matriz no TEST (se houver), senão no val
rep_true, rep_pred, rep_split = (
    (test_true, test_pred, 'test') if new_test_acc is not None else (val_true, val_pred, 'val')
)
print(f'\n=== Relatório no {rep_split} real (fine-tuned) ===')
print(classification_report(rep_true, rep_pred, target_names=short_names, zero_division=0))

cm = confusion_matrix(rep_true, rep_pred, labels=list(range(len(CLASS_NAMES))))
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=short_names, yticklabels=short_names,
            cmap='Greens', linewidths=0.5)
plt.xlabel('Predito'); plt.ylabel('Real')
plt.title(f'Fine-tuned no domínio real — {rep_split} {(new_test_acc or new_val_acc):.0%}')
plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0)
plt.tight_layout(); plt.show()

In [ ]:
# FT-5 — Salvar o modelo fine-tuned no Drive (só se o val real melhorou)
#
# Gate por VAL (seleção de modelo); o test fica como número honesto de relatório.
# Salvar com nome versionado evita sobrescrever o modelo bom em produção por engano.
from pathlib import Path
best_ft = Path(ft.trainer.best)
improved = new_val_acc is not None and new_val_acc > (base_val_acc or 0)

if improved:
    dest    = f'{SAVE_DIR}/soja_yolo11s_finetuned.pt'      # o que o app/Space carrega
    dest_v1 = f'{SAVE_DIR}/soja_yolo11s_finetuned_v1.pt'   # cópia versionada
    shutil.copy(best_ft, dest)
    shutil.copy(best_ft, dest_v1)
    print(f'✅ val melhorou ({base_val_acc:.1%} -> {new_val_acc:.1%}) — salvo em:')
    print(f'   {dest}')
    print(f'   {dest_v1}')
    if new_test_acc is not None:
        print(f'   (test honesto: {base_test_acc:.1%} -> {new_test_acc:.1%})')
    print('\nPróximo passo: subir esse .pt na raiz do HF Space soja-inspection-api (rebuild).')
else:
    print(f'❌ val NÃO melhorou ({base_val_acc:.1%} -> {new_val_acc:.1%}) — não salvei.')
    print('   Cheque: dataset balanceado (~100/classe)? captura no MESMO setup do uso real?')
    print('   Teste freeze=8 ou lr0 um pouco maior, e confirme que o crop está pegando o grão.')